# 01 — Main model: DINOv2 ViT-S/14 + LoRA (r=8) + MLP head + logit-adjusted loss, 5 seeds
Frozen self-supervised backbone, LoRA on attention q/k/v/out, fp16 AMP, no curriculum/cascade.
Resumable: if the kernel dies, just re-run the cells — finished seeds are skipped and an unfinished seed resumes from its last epoch.
Seed 42 is the **pre-declared primary checkpoint** (same protocol as paper 1). Do not pick a seed after seeing test results.

In [1]:
# ---- COMMON ----
# =============================================================================
# OralVision-P2  |  COMMON BLOCK (identical in every notebook)
# A Self-Supervised Vision Transformer Framework for Multi-Class Oral Disease Recognition
# DINOv2 ViT-S/14 + LoRA + MLP/cosine head + logit-adjusted loss (fp16)
# =============================================================================
import os
# Your env also has TensorFlow (broken protobuf). Tell transformers to ignore TF/Flax BEFORE it is imported.
os.environ["USE_TF"] = "0"; os.environ["USE_FLAX"] = "0"; os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

import gc, re, csv, json, math, time, random, copy, hashlib, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision
from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score,
                             cohen_kappa_score, classification_report)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# =============================================================================
# 1. PATHS  (EDIT ONLY THIS BLOCK IF YOUR FOLDERS DIFFER)
# =============================================================================
DESKTOP = Path(r"C:\Users\R Y Z E N\Desktop")

# --- same data as paper 1 (unchanged) ---
DATASET_ROOT     = DESKTOP / "oral_dataset_FINAL_CLEAN"        # contains train / val / test
RAW_DATA_DIR     = DESKTOP / "roboflow_dataset_merged"         # external, raw
BENGRAM_DATA_DIR = DESKTOP / "roboflow_bengram_illumination"   # external, Ben Graham

# --- all paper-2 outputs go here ---
P2_ROOT   = DESKTOP / "OralVision_P2"
OUT_00    = P2_ROOT / "00_setup_audit"
OUT_MAIN  = P2_ROOT / "01_main_5seed"
OUT_ABL   = P2_ROOT / "02_ablation"
OUT_EXT   = P2_ROOT / "03_external"
OUT_XAI   = P2_ROOT / "04_explainability"
OUT_STATS = P2_ROOT / "05_stats_complexity"

# --- paper-1 prediction files (only used by notebook 05; optional) ---
M4_INTERNAL_CSV = DESKTOP / "m4_statistical_rigor_report" / "test_predictions_all_models.csv"
M4_EXT_RAW_CSV  = DESKTOP / "external_eval_results_CLEAN_MODEL" / "external_eval_raw" / "predictions.csv"
M4_EXT_BG_CSV   = DESKTOP / "external_eval_results_CLEAN_MODEL" / "external_eval_bengram" / "predictions.csv"

for _d in (P2_ROOT, OUT_00, OUT_MAIN, OUT_ABL, OUT_EXT, OUT_XAI, OUT_STATS):
    _d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2. CONFIG
# =============================================================================
CLASS_NAMES = ["Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Tooth_Discoloration"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
SEEDS = [42, 123, 2024, 3407, 7777]          # same five seeds as paper 1
PRIMARY_SEED = 42

BACKBONES = {
    "dinov2_s14":  "facebook/dinov2-small",              # self-supervised (proposed)
    "vit_s16_sup": "WinKawaks/vit-small-patch16-224",    # supervised ImageNet ViT-S/16 (ablation)
    "deit_s16_sup": "facebook/deit-small-patch16-224",   # fallback supervised ViT-S/16
}

CFG = dict(
    img_size=224,
    batch_size=16, grad_accum=2,              # effective 32, same as paper 1 (RTX 2050 4 GB safe)
    num_workers=0, pin_memory=True,           # keep 0 on Windows/Jupyter
    backbone="dinov2_s14",
    use_lora=True, lora_r=8, lora_alpha=16, lora_dropout=0.0,
    lora_targets=("query", "key", "value", "out"),
    head_type="mlp",                          # "mlp" | "cosine" | "linear"
    head_hidden=512, head_dropout=0.1, cos_scale=30.0,
    loss="la",                                # "la" logit-adjusted | "ce" plain | "cw" class-weighted (paper 1 style)
    la_tau=1.0, label_smoothing=0.0,
    aug="photometric",                        # "photometric" | "basic" (paper 1 style)
    epochs=15, warmup_epochs=1, patience=5,
    lr_lora=3e-4, lr_head=1e-3, weight_decay=0.05, clip_grad=1.0,
    use_amp=True,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass


def gpu_report(prefix=""):
    if not torch.cuda.is_available():
        return
    free, total = torch.cuda.mem_get_info()
    log(f"{prefix}GPU alloc={torch.cuda.memory_allocated()/1e9:.2f} GB | "
        f"peak={torch.cuda.max_memory_allocated()/1e9:.2f} GB | free={free/1e9:.2f}/{total/1e9:.2f} GB")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))


# =============================================================================
# 3. DATA  (identical split discovery / sample ordering as paper 1)
# =============================================================================
def find_split_directory(root, names):
    if isinstance(names, str):
        names = [names]
    lowered = {x.lower() for x in names}
    cands = []
    for p in Path(root).rglob("*"):
        if p.is_dir() and p.name.lower() in lowered:
            class_dirs = [x for x in p.iterdir() if x.is_dir()]
            if len(class_dirs) >= 2:
                cands.append((len(class_dirs), p))
    if not cands:
        return None
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0][1]


def collect_samples(split_dir):
    samples = []
    for cname in CLASS_NAMES:
        cdir = Path(split_dir) / cname
        for path in sorted(cdir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXT:
                samples.append((str(path), CLASS_NAMES.index(cname)))
    return samples


def get_internal_splits():
    tr = find_split_directory(DATASET_ROOT, "train")
    va = find_split_directory(DATASET_ROOT, ["val", "validation"])
    te = find_split_directory(DATASET_ROOT, "test")
    if tr is None or va is None or te is None:
        raise FileNotFoundError(f"train/val/test folders not found under {DATASET_ROOT}")
    found = sorted(p.name for p in tr.iterdir() if p.is_dir())
    if found != sorted(CLASS_NAMES):
        raise RuntimeError(f"Class folders differ from expected.\nfound={found}\nexpected={sorted(CLASS_NAMES)}")
    return tr, va, te


class ImgDS(Dataset):
    """returns (image, label, index)"""
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i][0], self.samples[i][1]
        with Image.open(path) as im:
            im = im.convert("RGB")
        return self.transform(im), label, i


def collect_external_samples(root):
    """Same folder-name normalisation / duplicate-folder handling as paper 1's ExternalImageFolder.
    Returns list of (path, label, filename)."""
    canon = {c.lower(): c for c in CLASS_NAMES}
    seen, samples = {}, []
    for entry in sorted(os.listdir(root)):
        full = os.path.join(root, entry)
        if not os.path.isdir(full):
            continue
        c = canon.get(entry.strip().replace(" ", "_").lower())
        if c is None:
            log(f"  [WARN] unrecognised external folder '{entry}' skipped")
            continue
        if c in seen:
            log(f"  [NOTE] duplicate folder for '{c}' ('{entry}' vs '{seen[c]}') -> skipped duplicate")
            continue
        seen[c] = entry
        for fn in sorted(os.listdir(full)):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(full, fn), CLASS_NAMES.index(c), fn))
    if not samples:
        raise RuntimeError(f"No images found under {root}")
    return samples


# ----------------------------- augmentation ---------------------------------
class RandomColorConstancy:
    """Random re-illumination via Gray-World (p=1) / Shades-of-Gray (p=6) gains with random strength."""
    def __init__(self, p=0.5, minkowski=(1, 6), strength=(0.3, 1.0)):
        self.p, self.minkowski, self.strength = p, minkowski, strength

    def __call__(self, img):
        if random.random() > self.p:
            return img
        a = np.asarray(img, dtype=np.float32) / 255.0
        pn = random.choice(self.minkowski)
        e = np.power(np.power(a + 1e-6, pn).mean(axis=(0, 1)), 1.0 / pn)
        e = e / (np.linalg.norm(e) + 1e-8)
        gain = np.clip(1.0 / (e * math.sqrt(3.0) + 1e-8), 0.5, 2.0)
        gain = 1.0 + random.uniform(*self.strength) * (gain - 1.0)
        out = np.clip(a * gain, 0.0, 1.0)
        return Image.fromarray((out * 255.0).astype(np.uint8))


class RandomIlluminantShift:
    """Per-channel gain jitter (white-balance / illuminant colour change)."""
    def __init__(self, p=0.5, max_shift=0.12):
        self.p, self.m = p, max_shift

    def __call__(self, img):
        if random.random() > self.p:
            return img
        g = 1.0 + np.random.uniform(-self.m, self.m, size=3).astype(np.float32)
        a = np.asarray(img, dtype=np.float32) * g
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


class RandomGamma:
    def __init__(self, p=0.4, rng=(0.7, 1.5)):
        self.p, self.rng = p, rng

    def __call__(self, img):
        if random.random() > self.p:
            return img
        g = random.uniform(*self.rng)
        a = (np.asarray(img, dtype=np.float32) / 255.0) ** g
        return Image.fromarray((np.clip(a, 0, 1) * 255.0).astype(np.uint8))


def build_train_transform(kind="photometric", size=224):
    norm = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if kind == "basic":      # paper-1 style
        return transforms.Compose([
            transforms.Resize((size, size)), transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)] + norm)
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.75, 1.0), ratio=(0.9, 1.11)),
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
        RandomColorConstancy(0.5), RandomIlluminantShift(0.5), RandomGamma(0.4),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
        transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 1.5))], p=0.2)] + norm)


def build_eval_transform(size=224):
    return transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                               transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


def make_loader(ds, shuffle=False, batch_size=None, drop_last=False, generator=None):
    return DataLoader(ds, batch_size=batch_size or CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], drop_last=drop_last, generator=generator,
                      pin_memory=(CFG["pin_memory"] and DEVICE.type == "cuda"))


# =============================================================================
# 4. MODEL: frozen ViT + LoRA + head
# =============================================================================
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scaling = base, r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.lora_A.t() @ self.lora_B.t()) * self.scaling


_LORA_PATTERNS = {
    "query": re.compile(r"(^|\.)(query|q_proj)$"),
    "key":   re.compile(r"(^|\.)(key|k_proj)$"),
    "value": re.compile(r"(^|\.)(value|v_proj)$"),
    "out":   re.compile(r"(attention\.output\.dense|attn\.proj|attention\.out_proj|o_proj)$"),
}


def inject_lora(backbone, r, alpha, dropout, targets):
    to_wrap = []
    for name, mod in backbone.named_modules():
        if not isinstance(mod, nn.Linear):
            continue
        if ("attention" not in name) and ("attn" not in name):
            continue
        for t in targets:
            if _LORA_PATTERNS[t].search(name):
                to_wrap.append(name)
                break
    n_layers = backbone.config.num_hidden_layers
    if len(to_wrap) != n_layers * len(targets):
        raise RuntimeError(f"LoRA injection found {len(to_wrap)} linears, expected {n_layers*len(targets)}: {to_wrap[:8]}")
    for name in to_wrap:
        parent = backbone
        parts = name.split(".")
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], LoRALinear(getattr(parent, parts[-1]), r, alpha, dropout))
    return len(to_wrap)


def load_backbone(name, eager=False):
    from transformers import AutoModel
    kw = {"attn_implementation": "eager"} if eager else {}
    try:
        m = AutoModel.from_pretrained(BACKBONES[name], **kw)
    except Exception as e:
        if name == "vit_s16_sup":
            log(f"  [WARN] {BACKBONES[name]} failed ({type(e).__name__}); falling back to {BACKBONES['deit_s16_sup']}")
            m = AutoModel.from_pretrained(BACKBONES["deit_s16_sup"], **kw)
        else:
            raise
    return m


class CosineHead(nn.Module):
    def __init__(self, in_dim, num_classes, scale=30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_dim) * 0.02)
        self.scale = nn.Parameter(torch.tensor(float(scale)))

    def forward(self, x):
        return self.scale * F.linear(F.normalize(x.float(), dim=-1), F.normalize(self.weight.float(), dim=-1))


def build_head(head_type, in_dim, num_classes, hidden, dropout, cos_scale):
    if head_type == "linear":
        return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, num_classes))
    if head_type == "mlp":
        return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden), nn.GELU(),
                             nn.Dropout(dropout), nn.Linear(hidden, num_classes))
    if head_type == "cosine":
        return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden), nn.GELU(),
                             CosineHead(hidden, num_classes, cos_scale))
    raise ValueError(head_type)


class OralViT(nn.Module):
    """Backbone (frozen) [+ LoRA] -> [CLS ; mean-patch] -> head."""
    def __init__(self, cfg=None, eager=False):
        super().__init__()
        c = dict(CFG if cfg is None else cfg)
        self.cfg = c
        self.backbone = load_backbone(c["backbone"], eager=eager)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.n_lora = 0
        if c["use_lora"]:
            self.n_lora = inject_lora(self.backbone, c["lora_r"], c["lora_alpha"], c["lora_dropout"],
                                      tuple(c["lora_targets"]))
        dim = self.backbone.config.hidden_size
        self.feat_dim = 2 * dim
        self.head = build_head(c["head_type"], self.feat_dim, NUM_CLASSES, c["head_hidden"],
                               c["head_dropout"], c["cos_scale"])

    def encode(self, x):
        if self.cfg["use_lora"]:
            h = self.backbone(pixel_values=x).last_hidden_state
        else:
            with torch.no_grad():
                h = self.backbone(pixel_values=x).last_hidden_state
        return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=-1)

    def forward(self, x):
        return self.head(self.encode(x))


def trainable_state_dict(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


def load_model_from_ckpt(path, eager=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = OralViT(ck["cfg"], eager=eager)
    missing, unexpected = m.load_state_dict(ck["trainable_state"], strict=False)
    bad = [k for k in missing if ("lora" in k or k.startswith("head"))]
    if bad or unexpected:
        raise RuntimeError(f"Checkpoint mismatch. missing(trainable)={bad[:5]} unexpected={unexpected[:5]}")
    return m.to(DEVICE).eval(), ck


# =============================================================================
# 5. LOSS, METRICS
# =============================================================================
class LogitAdjustedCE(nn.Module):
    """Menon et al. 2021: add tau*log(prior) to logits at TRAIN time only; plain logits at inference."""
    def __init__(self, class_counts, tau=1.0, label_smoothing=0.0):
        super().__init__()
        prior = torch.tensor(class_counts, dtype=torch.float32)
        prior = prior / prior.sum()
        self.register_buffer("adj", tau * torch.log(prior))
        self.ls = label_smoothing

    def forward(self, logits, y):
        return F.cross_entropy(logits.float() + self.adj, y, label_smoothing=self.ls)


def build_criterion(loss_name, train_labels):
    counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    if loss_name == "la":
        return LogitAdjustedCE(counts, CFG["la_tau"], CFG["label_smoothing"]).to(DEVICE)
    if loss_name == "cw":
        from sklearn.utils.class_weight import compute_class_weight
        w = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
        return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE),
                                   label_smoothing=CFG["label_smoothing"])
    return nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])


def expected_calibration_error(prob, y, n_bins=15):
    conf, pred = prob.max(1), prob.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def compute_metrics(y, prob):
    prob = np.asarray(prob, dtype=np.float64)
    prob = prob / prob.sum(1, keepdims=True)
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(range(NUM_CLASSES)),
                                                 average=None, zero_division=0)
    try:
        auc = float(roc_auc_score(label_binarize(y, classes=list(range(NUM_CLASSES))), prob,
                                  average="macro", multi_class="ovr"))
    except Exception:
        auc = float("nan")
    return dict(
        n=int(len(y)), accuracy=float(accuracy_score(y, pred)),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)),
        macro_precision=float(p.mean()), macro_recall=float(r.mean()), macro_f1=float(f.mean()),
        macro_auc=auc, ece=expected_calibration_error(prob, y),
        qwk=float(cohen_kappa_score(y, pred, weights="quadratic")),
        per_class_precision=p.tolist(), per_class_recall=r.tolist(), per_class_f1=f.tolist(),
        tooth_disc_recall=float(r[CLASS_NAMES.index("Tooth_Discoloration")]),
        confusion=confusion_matrix(y, pred, labels=list(range(NUM_CLASSES))).tolist(),
    )


@torch.no_grad()
def predict_loader(model, loader, return_idx=False):
    model.eval()
    probs, ys, idxs = [], [], []
    for batch in loader:
        x, y = batch[0].to(DEVICE, non_blocking=True), batch[1]
        with autocast(device_type=DEVICE.type, enabled=CFG["use_amp"] and DEVICE.type == "cuda"):
            logits = model(x)
        probs.append(F.softmax(logits.float(), 1).cpu().numpy())
        ys.append(np.asarray(y))
        idxs.append(np.asarray(batch[2]))
    out = (np.concatenate(probs), np.concatenate(ys))
    return out + (np.concatenate(idxs),) if return_idx else out


def plot_confusion(cm, title, path, normalize=False):
    cm = np.array(cm)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    data = cm / cm.sum(1, keepdims=True).clip(min=1) if normalize else cm
    sns.heatmap(data, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def write_predictions_csv(path, samples, y, prob, with_path=True):
    pred = prob.argmax(1)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["image_index", "filename", "true_label", "pred_label", "correct"] +
                   [f"prob_{c}" for c in CLASS_NAMES])
        for i in range(len(y)):
            fn = os.path.basename(samples[i][0])
            w.writerow([i, fn, CLASS_NAMES[y[i]], CLASS_NAMES[pred[i]], int(y[i] == pred[i])] +
                       [f"{p:.6f}" for p in prob[i]])


# =============================================================================
# 6. TRAINING (resumable, per-epoch checkpoint)
# =============================================================================
def _run_training_impl(run_dir, seed, cfg_overrides=None, train_samples=None, val_samples=None, test_samples=None,
                 eval_train_at_end=False, tag=""):
    """Train one model. Skips if summary.json exists. Returns summary dict."""
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    sum_path = run_dir / "summary.json"
    if sum_path.exists():
        log(f"[skip] {run_dir.name} already finished")
        return json.load(open(sum_path))

    base_cfg = copy.deepcopy(CFG)
    if cfg_overrides:
        CFG.update(cfg_overrides)
    try:
        cfg = copy.deepcopy(CFG)
        seed_everything(seed)
        log("=" * 70); log(f"RUN {tag or run_dir.name} | seed={seed}"); log("=" * 70)
        log(f"cfg: " + json.dumps({k: cfg[k] for k in ('backbone', 'use_lora', 'lora_r', 'lora_targets', 'head_type',
                                                       'loss', 'aug', 'epochs')}, default=str))

        tr_ds = ImgDS(train_samples, build_train_transform(cfg["aug"], cfg["img_size"]))
        va_ds = ImgDS(val_samples, build_eval_transform(cfg["img_size"]))
        te_ds = ImgDS(test_samples, build_eval_transform(cfg["img_size"]))
        g = torch.Generator(); g.manual_seed(seed)
        tr_loader = make_loader(tr_ds, shuffle=True, drop_last=True, generator=g)
        va_loader, te_loader = make_loader(va_ds), make_loader(te_ds)
        train_labels = np.array([s[1] for s in train_samples])

        model = OralViT(cfg).to(DEVICE)
        tot, trn = count_params(model)
        log(f"params total={tot:,} trainable={trn:,} ({100*trn/tot:.2f}%)")
        criterion = build_criterion(cfg["loss"], train_labels)

        lora_p = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
        head_p = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
        groups = []
        if lora_p: groups.append(dict(params=lora_p, lr=cfg["lr_lora"], weight_decay=0.0))
        groups.append(dict(params=head_p, lr=cfg["lr_head"], weight_decay=cfg["weight_decay"]))
        opt = torch.optim.AdamW(groups)
        base_lrs = [gr["lr"] for gr in opt.param_groups]
        steps_per_epoch = len(tr_loader) // cfg["grad_accum"]
        total_steps = steps_per_epoch * cfg["epochs"]
        warm_steps = steps_per_epoch * cfg["warmup_epochs"]

        def set_lr(step):
            if step < warm_steps:
                f = (step + 1) / max(1, warm_steps)
            else:
                f = 0.5 * (1 + math.cos(math.pi * (step - warm_steps) / max(1, total_steps - warm_steps)))
            for gr, b in zip(opt.param_groups, base_lrs):
                gr["lr"] = b * max(f, 0.01)

        use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
        scaler = GradScaler("cuda", enabled=use_amp)

        start_epoch, best_f1, best_ep, bad, hist, gstep = 0, -1.0, -1, 0, [], 0
        last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
        if last_path.exists():
            ck = torch.load(last_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["trainable_state"], strict=False)
            opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
            start_epoch, best_f1, best_ep, bad, hist, gstep = (ck["epoch"] + 1, ck["best_f1"], ck["best_ep"],
                                                               ck["bad"], ck["hist"], ck["gstep"])
            log(f"Resumed from epoch {start_epoch} (best macro-F1 so far {best_f1:.4f})")

        t_start = time.time()
        for ep in range(start_epoch, cfg["epochs"]):
            model.train()
            if not cfg["use_lora"]:
                model.backbone.eval()
            t0, run_loss, n_seen, n_corr = time.time(), 0.0, 0, 0
            opt.zero_grad(set_to_none=True)
            for it, (x, y, _) in enumerate(tr_loader):
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                with autocast(device_type=DEVICE.type, enabled=use_amp):
                    logits = model(x)
                loss = criterion(logits.float(), y)
                scaler.scale(loss / cfg["grad_accum"]).backward()
                if (it + 1) % cfg["grad_accum"] == 0:
                    set_lr(gstep)
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_([p for gr in opt.param_groups for p in gr["params"]], cfg["clip_grad"])
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                    gstep += 1
                run_loss += loss.item() * len(y); n_seen += len(y)
                n_corr += (logits.argmax(1) == y).sum().item()
                if it % 200 == 0:
                    log(f"  ep{ep+1} it{it}/{len(tr_loader)} loss={run_loss/max(n_seen,1):.4f}")
            vprob, vy = predict_loader(model, va_loader)
            vm = compute_metrics(vy, vprob)
            vloss = float(-np.log(np.clip(vprob[np.arange(len(vy)), vy], 1e-8, 1)).mean())
            rec = dict(epoch=ep + 1, train_loss=run_loss / n_seen, train_acc=n_corr / n_seen, val_loss=vloss,
                       val_acc=vm["accuracy"], val_macro_f1=vm["macro_f1"], val_bal_acc=vm["balanced_accuracy"],
                       lr_head=opt.param_groups[-1]["lr"], sec=time.time() - t0)
            hist.append(rec)
            log(f"ep {ep+1}/{cfg['epochs']} | train_loss {rec['train_loss']:.4f} acc {rec['train_acc']:.4f} | "
                f"val_loss {vloss:.4f} acc {vm['accuracy']:.4f} macroF1 {vm['macro_f1']:.4f} | {rec['sec']:.0f}s | GPU {gpu_temp_clock()}")
            if vm["macro_f1"] > best_f1:
                best_f1, best_ep, bad = vm["macro_f1"], ep + 1, 0
                torch.save(dict(trainable_state=trainable_state_dict(model), cfg=cfg, epoch=ep + 1, seed=seed,
                                val_macro_f1=best_f1, class_names=CLASS_NAMES), best_path)
            else:
                bad += 1
            torch.save(dict(trainable_state=trainable_state_dict(model), opt=opt.state_dict(), scaler=scaler.state_dict(),
                            epoch=ep, best_f1=best_f1, best_ep=best_ep, bad=bad, hist=hist, gstep=gstep, cfg=cfg), last_path)
            if bad >= cfg["patience"]:
                log(f"Early stopping at epoch {ep+1} (best epoch {best_ep})")
                break
            cleanup_gpu()

        # ---- final evaluation with best checkpoint ----
        ck = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable_state"], strict=False)
        vprob, vy = predict_loader(model, va_loader)
        tprob, ty = predict_loader(model, te_loader)
        vm, tm = compute_metrics(vy, vprob), compute_metrics(ty, tprob)
        write_predictions_csv(run_dir / "val_predictions.csv", val_samples, vy, vprob)
        write_predictions_csv(run_dir / "test_predictions.csv", test_samples, ty, tprob)
        plot_confusion(tm["confusion"], f"Test confusion ({tag or run_dir.name}, seed {seed})", run_dir / "confusion_test.png")
        trm = None
        if eval_train_at_end:
            trprob, try_ = predict_loader(model, make_loader(ImgDS(train_samples, build_eval_transform(cfg["img_size"]))))
            trm = compute_metrics(try_, trprob)
        # curves
        try:
            import pandas as pd
            pd.DataFrame(hist).to_csv(run_dir / "history.csv", index=False)
            fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
            ax[0].plot([h["train_loss"] for h in hist], label="train"); ax[0].plot([h["val_loss"] for h in hist], label="val")
            ax[0].set_title("Loss"); ax[0].legend()
            ax[1].plot([h["val_macro_f1"] for h in hist]); ax[1].set_title("Val macro-F1")
            fig.tight_layout(); fig.savefig(run_dir / "curves.png", dpi=150); plt.close(fig)
        except Exception as e:
            log(f"  curves skipped: {e}")
        summary = dict(tag=tag or run_dir.name, seed=seed, cfg=cfg, best_epoch=best_ep, epochs_run=len(hist),
                       params_total=tot, params_trainable=trn, train_minutes=(time.time() - t_start) / 60.0,
                       val=vm, test=tm, train=trm, adapter_mb=os.path.getsize(best_path) / 1e6)
        save_json(summary, sum_path)
        log(f"DONE {run_dir.name}: test acc {tm['accuracy']:.4f} macroF1 {tm['macro_f1']:.4f} "
            f"TD-recall {tm['tooth_disc_recall']:.4f} (best ep {best_ep})")
        del model, opt; cleanup_gpu()
        return summary
    finally:
        CFG.clear(); CFG.update(base_cfg)



def run_training(run_dir, seed, cfg_overrides=None, *args, **kw):
    """RTX-2050 safe wrapper: on CUDA out-of-memory, halves batch_size and doubles grad_accum (effective batch unchanged)
    and resumes from the last finished epoch (at most one epoch is lost)."""
    ov = dict(cfg_overrides or {})
    bs, acc = ov.get("batch_size", CFG["batch_size"]), ov.get("grad_accum", CFG["grad_accum"])
    for attempt in range(4):
        cleanup_gpu()
        try:
            return _run_training_impl(run_dir, seed, {**ov, "batch_size": bs, "grad_accum": acc}, *args, **kw)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or bs <= 2:
                raise
            bs, acc = bs // 2, acc * 2
            log(f"[OOM] CUDA out of memory -> retrying with batch_size={bs}, grad_accum={acc} (effective batch kept), resuming from last epoch")
    raise RuntimeError("Out of memory even at batch_size=2")


def gpu_temp_clock():
    try:
        import subprocess
        o = subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu,clocks.sm,power.draw", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5).stdout.strip().split(",")
        return f"{o[0].strip()}C, {o[1].strip()}MHz, {o[2].strip()}W"
    except Exception:
        return "n/a"


def mean_sd(vals):
    v = np.array(vals, dtype=float)
    return float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0


log(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")
if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB)")


[21:58:48] torch 2.5.1+cu121 | torchvision 0.20.1+cu121 | device cuda
[21:58:48] GPU: NVIDIA GeForce RTX 2050 (4.29 GB)


In [2]:
import pandas as pd
tr, va, te = get_internal_splits()
TRAIN_S, VAL_S, TEST_S = collect_samples(tr), collect_samples(va), collect_samples(te)
log(f"train={len(TRAIN_S)} val={len(VAL_S)} test={len(TEST_S)}")
save_json(dict(CFG=CFG, seeds=SEEDS, primary=PRIMARY_SEED), OUT_MAIN / "config_used.json")

[21:59:40] train=17700 val=1265 test=1265


In [3]:
# ---- TRAIN ALL SEEDS (resumable; ~hours — see notebook 00 for your measured epoch time) ----
results = {}
for seed in SEEDS:
    results[seed] = run_training(OUT_MAIN / f"seed_{seed}", seed, None, TRAIN_S, VAL_S, TEST_S,
                                 eval_train_at_end=(seed == PRIMARY_SEED), tag="OralViT-P2 (full)")

[21:59:44] ======================================================================
[21:59:44] RUN OralViT-P2 (full) | seed=42
[21:59:44] ======================================================================
[21:59:44] cfg: {"backbone": "dinov2_s14", "use_lora": true, "lora_r": 8, "lora_targets": ["query", "key", "value", "out"], "head_type": "mlp", "loss": "la", "aug": "photometric", "epochs": 15}
[22:00:24] params total=22,749,830 trainable=693,254 (3.05%)
[22:00:28]   ep1 it0/1106 loss=1.3552
[22:02:02]   ep1 it200/1106 loss=1.1283
[22:03:14]   ep1 it400/1106 loss=0.7966
[22:04:26]   ep1 it600/1106 loss=0.6295
[22:05:35]   ep1 it800/1106 loss=0.5334
[22:06:48]   ep1 it1000/1106 loss=0.4647
[22:07:35] ep 1/15 | train_loss 0.4353 acc 0.8159 | val_loss 0.2470 acc 0.9162 macroF1 0.8274 | 431s | GPU 75C, 1830MHz, 26.79W
[22:07:38]   ep2 it0/1106 loss=0.0067
[22:08:47]   ep2 it200/1106 loss=0.1571
[22:09:59]   ep2 it400/1106 loss=0.1468
[22:11:11]   ep2 it600/1106 loss=0.1397
[22:12:22]   

In [4]:
# ---- AGGREGATE ----
results = {s: json.load(open(OUT_MAIN / f"seed_{s}" / "summary.json")) for s in SEEDS if (OUT_MAIN / f"seed_{s}" / "summary.json").exists()}
keys = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "macro_auc", "ece", "tooth_disc_recall"]
rows = []
for split in ("val", "test"):
    for k in keys:
        m, sd = mean_sd([results[s][split][k] for s in results])
        rows.append(dict(split=split, metric=k, mean=m, sd=sd, **{f"seed_{s}": results[s][split][k] for s in results}))
agg = pd.DataFrame(rows); agg.to_csv(OUT_MAIN / "seed_aggregate.csv", index=False)
print(agg[["split", "metric", "mean", "sd"]].to_string(index=False))

pc = pd.DataFrame({c: [results[s]["test"]["per_class_recall"][i] for s in results] for i, c in enumerate(CLASS_NAMES)}, index=list(results))
print("\nPer-class TEST recall by seed:\n", pc.round(4).to_string())
pc.loc["mean"], pc.loc["sd"] = pc.mean(), pc.std(ddof=1); pc.to_csv(OUT_MAIN / "per_class_recall_by_seed.csv")

P = results[PRIMARY_SEED]
lines = ["ORALVISION-P2 MAIN RESULTS", "=" * 60,
         f"Backbone: {P['cfg']['backbone']} | LoRA r={P['cfg']['lora_r']} targets={P['cfg']['lora_targets']} | head={P['cfg']['head_type']} | loss={P['cfg']['loss']}",
         f"Params total {P['params_total']:,} | trainable {P['params_trainable']:,} | adapter checkpoint {P['adapter_mb']:.2f} MB",
         f"Primary seed {PRIMARY_SEED}: best epoch {P['best_epoch']} | train time {P['train_minutes']:.1f} min", "",
         "Primary checkpoint (seed 42):"]
for split in ("train", "val", "test"):
    if P.get(split):
        d = P[split]; lines.append(f"  {split:5s} n={d['n']} acc={d['accuracy']:.4f} macroP={d['macro_precision']:.4f} macroR={d['macro_recall']:.4f} macroF1={d['macro_f1']:.4f}")
lines += ["", f"{len(results)}-seed mean ± SD (TEST):"]
for k in keys:
    m, sd = mean_sd([results[s]['test'][k] for s in results]); lines.append(f"  {k:20s} {m:.4f} ± {sd:.4f}")
lines += ["", "Train minutes per seed: " + ", ".join(f"{s}:{results[s]['train_minutes']:.0f}" for s in results)]
open(OUT_MAIN / "main_report.txt", "w", encoding="utf-8").write("\n".join(lines)); print("\n".join(lines))
# per-class table of primary checkpoint
tm = P["test"]
print("\nPrimary checkpoint per-class TEST:")
print(pd.DataFrame(dict(precision=tm["per_class_precision"], recall=tm["per_class_recall"], f1=tm["per_class_f1"],
                        support=np.array(tm["confusion"]).sum(1)), index=CLASS_NAMES).round(4).to_string())
plot_confusion(tm["confusion"], "OralViT-P2 internal test (seed 42)", OUT_MAIN / "confusion_test_primary.png")

split            metric     mean       sd
  val          accuracy 0.983557 0.004008
  val balanced_accuracy 0.967713 0.011871
  val   macro_precision 0.967127 0.015055
  val      macro_recall 0.967713 0.011871
  val          macro_f1 0.967017 0.013266
  val         macro_auc 0.999129 0.000424
  val               ece 0.013149 0.003458
  val tooth_disc_recall 0.907692 0.051602
 test          accuracy 0.979921 0.005379
 test balanced_accuracy 0.940357 0.017846
 test   macro_precision 0.956181 0.018200
 test      macro_recall 0.940357 0.017846
 test          macro_f1 0.945779 0.013493
 test         macro_auc 0.998702 0.000825
 test               ece 0.015967 0.004180
 test tooth_disc_recall 0.753846 0.107417

Per-class TEST recall by seed:
       Calculus  Caries  Gingivitis  Hypodontia  Mouth_Ulcer  Tooth_Discoloration
42      1.0000  0.9948      0.9839      0.9835       0.9928               0.7308
123     0.9403  0.9767      0.9731      0.9890       1.0000               0.6538
2024    0.